# ASSIGNMENT  NLP- 2 (Sentiment Analysis )


## Assignment: Sentiment Analysis using NLP Pipeline & ML Models

**1. Data Understanding**

In [ ]:
import pandas as pd
import kagglehub
import os

# Download dataset
folder_path = kagglehub.dataset_download("a/twitter-tweets-sentiment-dataset")

print("Files in dataset:", os.listdir(folder_path))

csv_path = os.path.join(folder_path, "Tweets.csv")
df = pd.read_csv(csv_path)

# Basic info
print("Dataset shape:", df.shape)
print("Total reviews:", len(df))

# Step 4: Sentiment distribution
print("\nSentiment counts:")
print(df['sentiment'].value_counts())

# Step 5: Example reviews
print("\nPositive example:")
print(df[df['sentiment'] == 'positive']['text'].iloc[0])

print("\nNegative example:")
print(df[df['sentiment'] == 'negative']['text'].iloc[0])

Using Colab cache for faster access to the 'twitter-tweets-sentiment-dataset' dataset.
Files in dataset: ['Tweets.csv']
Dataset shape: (27481, 4)
Total reviews: 27481

Sentiment counts:
sentiment
neutral     11118
positive     8582
negative     7781
Name: count, dtype: int64

Positive example:
2am feedings for the baby are fun when he is all smiles and coos

Negative example:
 Sooo SAD I will miss you here in San Diego!!!


**2. NLP Preprocessing**

In [ ]:
import pandas as pd
import numpy as np
import re
import string

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
# Lowercase
df['text'] = df['text'].str.lower()

# Remove URLs
def remove_urls(text):
    return re.sub(r"http\S+|www\S+", "", text)

# Remove special characters
def remove_special_chars(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

# Remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

# Tokenization
def tokenize(text):
    return word_tokenize(text)

# Remove stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

# Stemming
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def stemming(tokens):
    return [stemmer.stem(word) for word in tokens]

def lemmatization(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

# Combine all steps into one function
def clean_text(text):
    if pd.isna(text):
      return ""
    text = text.lower()
    text = remove_urls(text)
    text = remove_special_chars(text)
    text = remove_punctuation(text)

    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatization(tokens)

    return " ".join(tokens)

In [ ]:
# Test the Function
sample_review = "oh Marly, I`m so sorry!! I hope you find her soon!! <3 <3, check in: https://twitter.com"
print("\n----- Testing the Cleaning Function -----")
print("Original Text:", sample_review)
print("Cleaned Text :", clean_text(sample_review))

print("\nCleaning the entire dataset")
df['cleaned_text'] = df['text'].apply(clean_text)

print("\n----- First 5 rows of Cleaned Data -----")
df[['text', 'cleaned_text']].head()


----- Testing the Cleaning Function -----
Original Text: oh Marly, I`m so sorry!! I hope you find her soon!! <3 <3, check in: https://twitter.com
Cleaned Text : oh marly im sorry hope find soon check

Cleaning the entire dataset

----- First 5 rows of Cleaned Data -----


,text,cleaned_text
0,"i`d have responded, if i were going",id responded going
1,sooo sad i will miss you here in san diego!!!,sooo sad miss san diego
2,my boss is bullying me...,bos bullying
3,what interview! leave me alone,interview leave alone
4,"sons of ****, why couldn`t they put them on t...",son couldnt put release already bought


**3. Feature Engineering**

In [ ]:
#Bag of Words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['cleaned_text'])
print("BoW Shape:", X_bow.shape)

#TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['cleaned_text'])
print("TF-IDF Shape:", X_tfidf.shape)

# Words learned by vectorizers
print("\nSample of learned vocabulary (first 20 words):")
vocabulary = bow.get_feature_names_out()
print(vocabulary[:20])

print("\nSample of TF-IDF vocabulary:")
tfidf_vocab = tfidf.get_feature_names_out()
print(tfidf_vocab[:20])


# Convert sentiment labels to numbers
print("\nConverting sentiment labels to numbers...")
df['sentiment_numeric'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0,
    'neutral': 2
})

y = df['sentiment_numeric'].values

BoW Shape: (27481, 5000)
TF-IDF Shape: (27481, 5000)

Sample of learned vocabulary (first 20 words):
['aaaah' 'aah' 'aaron' 'ab' 'abandoned' 'abby' 'ability' 'abit' 'able'
 'absolutely' 'abt' 'ac' 'academy' 'accept' 'accepted' 'accepting'
 'access' 'accident' 'accidentally' 'accomplished']

Sample of TF-IDF vocabulary:
['aaaah' 'aah' 'aaron' 'ab' 'abandoned' 'abby' 'ability' 'abit' 'able'
 'absolutely' 'abt' 'ac' 'academy' 'accept' 'accepted' 'accepting'
 'access' 'accident' 'accidentally' 'accomplished']

Converting sentiment labels to numbers...


**4. Model Building**

In [ ]:
# Splitting data into Training and Testing sets
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]} reviews")
print(f"Testing set size: {X_test.shape[0]} reviews")

log_model=LogisticRegression(max_iter=1000)
nb_model = MultinomialNB()
tree_model = DecisionTreeClassifier(random_state=42)

# Logistic Regression
log_model.fit(X_train, y_train)

# Naive Bayes
nb_model.fit(X_train, y_train)

# Decision Tree
tree_model.fit(X_train, y_train)

print("\nAll models have been successfully trained!")

Training set size: 21984 reviews
Testing set size: 5497 reviews

All models have been successfully trained!


**5. Model Evaluation**

In [ ]:
print("Starting Model Evaluation...\n")

log_predictions = log_model.predict(X_test)
nb_predictions = nb_model.predict(X_test)
tree_predictions = tree_model.predict(X_test)


def evaluate_model(y_true, y_pred, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall = recall_score(y_true, y_pred, average='weighted')
    f1 = f1_score(y_true, y_pred, average='weighted')

    return {
        'Model': model_name,
        'Accuracy': round(accuracy, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1 Score': round(f1, 4)
    }


# Calculate Scores
log_results = evaluate_model(y_test, log_predictions, 'Logistic Regression')
print(log_results)
nb_results = evaluate_model(y_test, nb_predictions, 'Naive Bayes')
print(nb_results)
tree_results = evaluate_model(y_test, tree_predictions, 'Decision Tree')
print(tree_results)

Starting Model Evaluation...

{'Model': 'Logistic Regression', 'Accuracy': 0.6831, 'Precision': 0.6936, 'Recall': 0.6831, 'F1 Score': 0.6831}
{'Model': 'Naive Bayes', 'Accuracy': 0.6367, 'Precision': 0.6616, 'Recall': 0.6367, 'F1 Score': 0.6346}
{'Model': 'Decision Tree', 'Accuracy': 0.6383, 'Precision': 0.6375, 'Recall': 0.6383, 'F1 Score': 0.6376}


**6. Comparison & Insights**

In [ ]:
all_results = [log_results, nb_results, tree_results]
comparison_df = pd.DataFrame(all_results)

print("\n----- Final Model Comparison -----")
comparison_df


----- Final Model Comparison -----


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.6831,0.6936,0.6831,0.6831
1,Naive Bayes,0.6367,0.6616,0.6367,0.6346
2,Decision Tree,0.6383,0.6375,0.6383,0.6376


# Comparison & Insight
## Best Preprocessing
**Lemmatization is better than Stemming**
Lemmatization gives real words
Example:

1. better → good

2. Stemming: better → better or weird forms

**Remove unwanted data (noise)**

Delete:

* HTML tags and
common words like the, is, and

* This helps the model focus only on important words like good, bad, amazing

---

## Best Vectorization

**TF-IDF is better than Bag of Words (BoW)**

BoW:

* Just counts words

* Treats all words equally

TF-IDF:

* Gives importance to meaningful words

* Reduces importance of common words

Example:

1. “movie” → less important

2. “fantastic” → more important

---

## Best Model

**Logistic Regression is the best here**

Why?

* Works very well with text data

* Finds a clear boundary between positive and negative words